# 04 — Riding Style Scoring

Compute a smoothness score (0–100) from the jerk of `GYR_Y_DEG`.
Higher jerk = more aggressive steering. Use as a proxy for rider behaviour.

In [1]:
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

from scooter_imu.data.loader import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.riding_style import RidingStyleScorer

RAW_PATH = Path("../data/raw/ScooterIMUData.csv")

In [2]:
df = preprocess(load_raw(RAW_PATH))
result = RidingStyleScorer().predict(df)
print(f"Overall smooth score: {result['smooth_score'].mean():.1f} / 100")
result[["timestamp", "gyr_y", "jerk", "aggression", "smooth_score"]].head()

Overall smooth score: 78.1 / 100


,timestamp,gyr_y,jerk,aggression,smooth_score
0,2024-03-15 10:00:00.000,-0.001133,0.00000,0.000000,100.0
1,2024-03-15 10:00:00.020,-0.001249,-0.00580,0.001102,99.9
2,2024-03-15 10:00:00.040,0.002650,0.19495,0.035207,96.5
3,2024-03-15 10:00:00.060,0.000661,-0.09945,0.036125,96.4
4,2024-03-15 10:00:00.080,-0.000423,-0.05420,0.035170,96.5


In [3]:
fig = px.line(result, x="timestamp", y="smooth_score",
              title="Smooth Score Over Trip",
              labels={"smooth_score": "Smooth Score (0–100)"})
fig.add_hline(y=70, line_dash="dash", line_color="green", annotation_text="Good")
fig.show()

In [4]:
fig = px.histogram(result, x="jerk", nbins=120,
                   title="Jerk Distribution",
                   labels={"jerk": "Jerk (deg/s²)"})
fig.show()

In [5]:
# Jerk vs GYR_Y scatter
fig = px.scatter(result, x="gyr_y", y="jerk", color="smooth_score",
                 color_continuous_scale="RdYlGn",
                 title="Jerk vs GYR_Y coloured by Smooth Score")
fig.show()

## Rider Dashboard — Smooth Score Gauge

The needle reflects overall trip smoothness.
**Green zone** (70–100) = smooth rider · **Orange** (40–70) = moderate · **Red** (0–40) = aggressive.
The green threshold line at 70 marks the "good rider" boundary.

In [6]:
import plotly.graph_objects as go

overall = result["smooth_score"].mean()

fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=overall,
    delta=dict(reference=70, valueformat=".1f",
               increasing=dict(color="#44cc66"),
               decreasing=dict(color="#cc2200")),
    number=dict(suffix=" / 100", font=dict(size=32, color="#1e3a5f")),
    title=dict(text="Overall Smooth Score", font=dict(size=18, color="#1e3a5f")),
    gauge=dict(
        axis=dict(range=[0, 100], tickwidth=1, tickcolor="#888",
                  tickvals=[0, 20, 40, 60, 70, 80, 100],
                  ticktext=["0", "20", "40", "60", "Good↑", "80", "100"]),
        bar=dict(color="#2E86AB", thickness=0.28),
        bgcolor="white",
        borderwidth=2,
        bordercolor="#ccc",
        steps=[
            dict(range=[0,  40], color="#ffcccc"),
            dict(range=[40, 70], color="#fff0cc"),
            dict(range=[70, 100], color="#ccffdd"),
        ],
        threshold=dict(line=dict(color="#006633", width=5), thickness=0.8, value=70),
    ),
))
fig.update_layout(
    height=360,
    paper_bgcolor="#f8fbff",
    font=dict(family="Arial"),
    margin=dict(l=30, r=30, t=60, b=30),
)
fig.show()

## Live Gauge — Score Evolving Over the Trip

The needle updates frame-by-frame as the scooter moves through the trip.
Aggressive moments (sharp turns, sudden inputs) dip the score toward red.

In [7]:
import numpy as np
import plotly.graph_objects as go

scores     = result["smooth_score"].to_numpy()
timestamps = result["timestamp"].astype(str).str[:19].to_numpy()

n_frames = min(100, len(scores))
idx = np.linspace(0, len(scores) - 1, n_frames, dtype=int)

def _bar_color(s):
    if s >= 70: return "#2ecc71"
    if s >= 40: return "#f39c12"
    return "#e74c3c"

gauge_steps = [
    dict(range=[0,  40], color="#ffcccc"),
    dict(range=[40, 70], color="#fff0cc"),
    dict(range=[70, 100], color="#ccffdd"),
]

frames = [
    go.Frame(
        data=[go.Indicator(
            mode="gauge+number",
            value=float(scores[idx[k]]),
            number=dict(suffix=" / 100", font=dict(size=28)),
            gauge=dict(
                axis=dict(range=[0, 100]),
                bar=dict(color=_bar_color(scores[idx[k]]), thickness=0.3),
                steps=gauge_steps,
                threshold=dict(line=dict(color="#006633", width=4), thickness=0.8, value=70),
            ),
            title=dict(text=f"Smooth Score  —  {timestamps[idx[k]]}", font=dict(size=13)),
        )],
        name=str(k),
    )
    for k in range(n_frames)
]

fig = go.Figure(
    data=[go.Indicator(
        mode="gauge+number",
        value=float(scores[0]),
        number=dict(suffix=" / 100", font=dict(size=28)),
        gauge=dict(
            axis=dict(range=[0, 100]),
            bar=dict(color="#2E86AB", thickness=0.3),
            steps=gauge_steps,
            threshold=dict(line=dict(color="#006633", width=4), thickness=0.8, value=70),
        ),
        title=dict(text="Smooth Score — press ▶ Play to animate", font=dict(size=13)),
    )],
    layout=go.Layout(
        height=400,
        paper_bgcolor="#f8fbff",
        margin=dict(l=30, r=30, t=30, b=60),
        updatemenus=[dict(
            type="buttons", showactive=False, y=-0.14, x=0.5, xanchor="center",
            buttons=[
                dict(label="▶  Play",  method="animate",
                     args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True)]),
                dict(label="⏸  Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            steps=[dict(args=[[f.name], dict(frame=dict(duration=0), mode="immediate")],
                        method="animate", label="") for f in frames],
            active=0, y=-0.04, len=0.9, x=0.05, currentvalue=dict(visible=False),
        )],
    ),
    frames=frames,
)
fig.show()